In [1]:
import pandas as pd
import pdfplumber
import re

In [2]:
PDF_PATH = 'ACT-Workers-Compensation-Suggested-Reasonable-Premium-Rates-2025-26.pdf'

pdf = pdfplumber.open(PDF_PATH)
print(f'Total pages: {len(pdf.pages)}')

Total pages: 4


In [3]:
# Extract rows from text (no table boundaries detected by pdfplumber).
# Each data line follows this pattern:
#   <4-digit ANZSIC> <description> <wages $m> <freq rel> <cost rel>
#   <2024/25 relativity> <2025/26 relativity> <2024/25 rate%> <2025/26 rate%>
#
# The last column ("Suggested Premium Rate") is the capped 2025/26 rate.
# ACT uses ANZSIC codes directly (no separate WIC codes).
# Not all 506 ANZSIC classes appear — ACT only lists classes with
# material employer wages in the territory.

pattern = re.compile(
    r'^(\d{4})\s+'           # ANZSIC code
    r'(.+?)\s+'              # Description
    r'([\d,.]+)\s+'          # Estimated wages ($m)
    r'([\d,.]+)\s+'          # Claim freq rel (last 3 yrs)
    r'([\d,.]+)\s+'          # Claim cost rel (last 5 yrs)
    r'([\d,.]+)\s+'          # 2024/25 selected relativity
    r'([\d,.]+)\s+'          # 2025/26 selected relativity
    r'([\d.]+)%\s+'          # 2024/25 suggested premium rate
    r'([\d.]+)%\s*$'         # 2025/26 suggested premium rate (capped)
)

rows = []
for pg_idx in range(len(pdf.pages)):
    text = pdf.pages[pg_idx].extract_text()
    for line in text.strip().split('\n'):
        m = pattern.match(line.strip())
        if m:
            rows.append({
                'anzsic_code': m.group(1),
                'description': m.group(2).strip(),
                'estimated_wages_millions': float(m.group(3).replace(',', '')),
                'suggested_premium_rate_pct': float(m.group(9)),
                'prior_year_rate_pct': float(m.group(8)),
            })

df = pd.DataFrame(rows)
print(f'{len(df)} rows parsed')
df.head()

352 rows parsed


,anzsic_code,description,estimated_wages_millions,suggested_premium_rate_pct,prior_year_rate_pct
0,0112,Nursery Production (Outdoors),1.5,3.78,3.85
1,0113,Turf Growing,1.6,3.75,3.50
2,0123,Vegetable Growing (Outdoors),0.1,3.65,3.42
3,0141,Sheep Farming (Specialised),0.1,4.48,4.38
4,0144,Sheep-Beef Cattle Farming,0.9,4.63,4.55


In [4]:
df[['suggested_premium_rate_pct', 'prior_year_rate_pct']].describe()

,suggested_premium_rate_pct,prior_year_rate_pct
count,352.000000,352.000000
mean,3.694801,3.416847
std,2.924530,2.352270
min,0.260000,0.270000
25%,1.687500,1.627500
50%,3.020000,2.990000
75%,4.612500,4.442500
max,32.460000,17.040000


In [5]:
# Spot-check first, last, and high-rate rows
spot_checks = ['0112', '9601', '9129', '3222']
df[df['anzsic_code'].isin(spot_checks)][['anzsic_code', 'description', 'suggested_premium_rate_pct']]

,anzsic_code,description,suggested_premium_rate_pct
0,0112,Nursery Production (Outdoors),3.78
100,3222,Bricklaying Services,12.19
325,9129,Other Horse and Dog Racing Activities,32.46
351,9601,Private Households Employing Staff,2.72


In [6]:
df.shape

(352, 5)

In [7]:
df.to_parquet('ACT_WIC.parquet', index=False)
print(f'Saved ACT_WIC.parquet ({len(df)} rows)')

Saved ACT_WIC.parquet (352 rows)
